# Do you know when you know? — reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alestainer/statistics-intuitions/blob/main/notebooks/03-function-sampling.ipynb)

Reproduces the fixed 20-round function-sampling benchmark. It loads the exact formulas, reveal order, plot images, prompts and saved model decisions used in the article. No paid request runs unless `RUN_MODEL` is set explicitly.


In [ ]:
from pathlib import Path
import json, urllib.request

RAW = "https://raw.githubusercontent.com/Alestainer/statistics-intuitions/main/"

def load_json(relative_path):
    candidates = [Path("../") / relative_path, Path(relative_path)]
    for path in candidates:
        if path.exists():
            return json.loads(path.read_text())
    with urllib.request.urlopen(RAW + relative_path) as response:
        return json.load(response)

benchmark = load_json("data/03-function-sampling/benchmark.json")
results = load_json("data/03-function-sampling/results.json")
print(f"{len(benchmark['rounds'])} rounds; {len(results['models'])} completed model runs")


## Inspect the exact sampling sequence

In [ ]:
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

VARIANT_TEXT = {
    "linear-1": "f(x) = x", "linear-3": "f(x) = 3x",
    "square-1": "f(x) = x²", "square-4": "f(x) = 4x²",
    "log-1": "f(x) = ln(1 + x)", "log-4": "f(x) = ln(1 + 4x)",
    "reciprocal-1": "f(x) = 1 / x", "reciprocal-3": "f(x) = 3 / x",
    "hump-1": "f(x) = 4x(1 − x)", "hump-2": "f(x) = 8x(1 − x)",
    "valley-1": "f(x) = |2x − 1|", "valley-3": "f(x) = 3|2x − 1|",
    "sine-2pi": "f(x) = sin(2πx)", "sine-4pi": "f(x) = 2sin(4πx)",
    "plateau-1": "f(x) = min(2x, 1)", "plateau-3": "f(x) = 3 min(2x, 1)",
}

def read_bytes(relative_path):
    for path in (Path("../") / relative_path, Path(relative_path)):
        if path.exists(): return path.read_bytes()
    with urllib.request.urlopen(RAW + relative_path) as response: return response.read()

def state_image(round_number, samples):
    path = f"data/03-function-sampling/stimuli/round-{round_number:02d}/samples-{samples:02d}.png"
    return Image.open(BytesIO(read_bytes(path))).convert("RGB")

round_number = 1
round_data = benchmark["rounds"][round_number - 1]
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for samples, ax in enumerate(axes, start=1):
    ax.imshow(state_image(round_number, samples)); ax.axis("off"); ax.set_title(f"{samples} sample(s)")
plt.show()
for card in round_data["cards"]:
    print(f'{card["label"]}: {VARIANT_TEXT[card["variantId"]]}')
print("Expected:", "NOT DISTINGUISHABLE" if len(round_data["hiddenLabels"]) > 1 else "OPTION " + round_data["hiddenLabels"][0])


## Exact model prompt

In [ ]:
SYSTEM_PROMPT = """You are playing a visual function-identification game. A plot shows the currently revealed observations; four candidate formulas are listed in the accompanying text. Axes have no fixed units: identify qualitative shape, not coefficient scale. Return exactly one command and nothing else: OPTION A, OPTION B, OPTION C, OPTION D, SAMPLE MORE, or NOT DISTINGUISHABLE.

Use SAMPLE MORE when current evidence is insufficient but another observation could separate the options. Use NOT DISTINGUISHABLE only when multiple options represent the same qualitative shape under independent positive axis scaling, so no future observation can uniquely separate them. At most six samples are available."""

def state_text(round_data, samples):
    cards = "\n".join(f'{c["label"]}: {VARIANT_TEXT[c["variantId"]]}' for c in round_data["cards"])
    return f"Visible samples: {samples} of 6.\n{cards}\nReturn exactly one valid command."

print(SYSTEM_PROMPT)
print("\n--- example user message ---\n")
print(state_text(round_data, 3))


## Recompute the published table

In [ ]:
import pandas as pd

rows = []
for run in results["models"]:
    rounds = run["rounds"]
    correct = [r for r in rounds if r["correct"]]
    false_nonunique = sum(r["finalCommand"] == "NOT DISTINGUISHABLE" and r["expectedCommand"] != "NOT DISTINGUISHABLE" for r in rounds)
    rows.append({
        "model": run["model"],
        "correct": f'{len(correct)}/20',
        "mean samples on correct": round(sum(r["samplesUsed"] for r in correct) / len(correct), 2),
        "false non-unique calls": false_nonunique,
        "cost (USD)": round(sum(r["costUsd"] for r in rounds), 4),
    })
pd.DataFrame(rows).sort_values(["correct", "mean samples on correct"], ascending=[False, True]).reset_index(drop=True)


## Optional paid rerun

The frozen images and prompt above are sufficient to rerun any round through a vision API. Calls are deliberately omitted from automatic execution: model identifiers, providers, prices and outputs can change. Use the exact image returned by `state_image`, send `SYSTEM_PROMPT` as the system message and `state_text(...)` plus the PNG as the user message. Start at one sample; only request the next state when the reply is exactly `SAMPLE MORE`.
